<a href="https://colab.research.google.com/github/sunnyraavan6-Sunny/bank-intent-slm/blob/main/bank_intent_slm_ipynb_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!nvidia-smi

Mon Oct 12 07:56:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q -U transformers datasets peft trl accelerate bitsandbytes evaluate scikit-learn gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 77.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 11.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow

In [3]:
import torch, transformers, peft, trl, datasets
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("trl:", trl.__version__)
print("datasets:", datasets.__version__)

CUDA available: True
GPU: Tesla T4
transformers: 5.19.0
peft: 0.21.2
trl: 1.15.0
datasets: 5.1.0


In [4]:
from google.colab import userdata
from huggingface_hub import login, whoami

login(token=userdata.get("HF_TOKEN"))
print("Logged in as:", whoami()["name"])

Logged in as: Kishorekumar32


In [6]:
from datasets import load_dataset

ds = load_dataset("mteb/banking77")
print(ds)
print()
print("Columns:", ds["train"].column_names)
print("Example row:", ds["train"][0])

README.md:   0%|          | 0.00/16.6k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  294kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 91.4kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/9993 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3076 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 9993
    })
    test: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 3076
    })
})

Columns: ['text', 'label', 'label_text']
Example row: {'text': 'I am still waiting on my card?', 'label': 11, 'label_text': 'card_arrival'}


In [7]:
label_names = sorted(set(ds["train"]["label_text"]))
print("Number of intents:", len(label_names))
print("First 10:", label_names[:10])

Number of intents: 77
First 10: ['Refund_not_showing_up', 'activate_my_card', 'age_limit', 'apple_pay_or_google_pay', 'atm_support', 'automatic_top_up', 'balance_not_updated_after_bank_transfer', 'balance_not_updated_after_cheque_or_cash_deposit', 'beneficiary_not_allowed', 'cancel_transfer']


In [8]:
import pandas as pd

train_df = ds["train"].to_pandas()
test_df  = ds["test"].to_pandas()
print("Shapes:", train_df.shape, test_df.shape)
print(train_df.head(3))

# 1. Missing / empty
print("\nMissing values:\n", train_df.isna().sum())
print("Empty texts:", (train_df["text"].str.strip() == "").sum())

# 2. Duplicates
print("\nDuplicate texts in train:", train_df.duplicated("text").sum())

# 3. Conflicting labels
labels_per_text = train_df.groupby("text")["label_text"].nunique()
print("Texts with more than one label:", (labels_per_text > 1).sum())

# 4. Leakage between train and test
overlap = set(train_df["text"]) & set(test_df["text"])
print("Train/test overlap:", len(overlap))

# 5. Class balance
counts = train_df["label_text"].value_counts()
print("\nExamples per intent:\n", counts.describe())
print("Most common:\n", counts.head(3))
print("Least common:\n", counts.tail(3))

# 6. Text length (in words)
train_df["n_words"] = train_df["text"].str.split().str.len()
print("\nWords per message:\n", train_df["n_words"].describe())

Shapes: (9993, 3) (3076, 3)
                                                text  label    label_text
0                     I am still waiting on my card?     11  card_arrival
1  What can I do if my card still hasn't arrived ...     11  card_arrival
2  I have been waiting over a week. Is the card s...     11  card_arrival

Missing values:
 text          0
label         0
label_text    0
dtype: int64
Empty texts: 0

Duplicate texts in train: 0
Texts with more than one label: 0
Train/test overlap: 0

Examples per intent:
 count     77.000000
mean     129.779221
std       32.974321
min       35.000000
25%      111.000000
50%      127.000000
75%      158.000000
max      187.000000
Name: count, dtype: float64
Most common:
 label_text
card_payment_fee_charged                            187
direct_debit_payment_not_recognised                 182
balance_not_updated_after_cheque_or_cash_deposit    181
Name: count, dtype: int64
Least common:
 label_text
card_acceptance             59
virtual_ca

In [9]:
import json
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict

# 1. Stratified train/validation split
base = train_df[["text", "label_text"]]
tr_df, val_df = train_test_split(
    base, test_size=0.1, random_state=42, stratify=base["label_text"]
)
te_df = test_df[["text", "label_text"]]
print("Train / Val / Test:", len(tr_df), len(val_df), len(te_df))

# 2. Chat format
SYSTEM = (
    "You are a banking assistant. Classify the customer's message into one "
    "intent. Reply only with JSON in the form {\"intent\": \"<intent_name>\"}."
)

def to_chat(row):
    return {
        "messages": [
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": row["text"]},
            {"role": "assistant", "content": json.dumps({"intent": row["label_text"]})},
        ],
        "intent": row["label_text"],   # kept for scoring later
        "text": row["text"],
    }

def build(df):
    return Dataset.from_list([to_chat(r) for r in df.to_dict("records")])

data = DatasetDict({
    "train": build(tr_df),
    "validation": build(val_df),
    "test": build(te_df),
})
print(data)
print("\nOne example:")
for m in data["train"][0]["messages"]:
    print(f"[{m['role']}] {m['content']}")

Train / Val / Test: 8993 1000 3076
DatasetDict({
    train: Dataset({
        features: ['messages', 'intent', 'text'],
        num_rows: 8993
    })
    validation: Dataset({
        features: ['messages', 'intent', 'text'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['messages', 'intent', 'text'],
        num_rows: 3076
    })
})

One example:
[system] You are a banking assistant. Classify the customer's message into one intent. Reply only with JSON in the form {"intent": "<intent_name>"}.
[user] how do i setup my apple pay
[assistant] {"intent": "apple_pay_or_google_pay"}


In [10]:
import torch, json, re
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL, padding_side="left")
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16, device_map="cuda")
model.eval()

label_names = sorted(set(train_df["label_text"]))
BASE_SYSTEM = SYSTEM + "\nValid intents: " + ", ".join(label_names)

def make_prompt(text, system):
    msgs = [{"role": "system", "content": system},
            {"role": "user", "content": text}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def parse_intent(s):
    m = re.search(r"\{.*?\}", s, re.S)
    if not m:
        return None
    try:
        return json.loads(m.group())["intent"]
    except Exception:
        return None

@torch.no_grad()
def predict(texts, system, bs=16):
    outs = []
    for i in range(0, len(texts), bs):
        prompts = [make_prompt(t, system) for t in texts[i:i+bs]]
        enc = tok(prompts, return_tensors="pt", padding=True).to("cuda")
        gen = model.generate(**enc, max_new_tokens=30, do_sample=False)
        outs += tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return outs

# Fixed 200-sample test subset (same one we'll reuse after fine-tuning)
sub = data["test"].shuffle(seed=42).select(range(200))
texts, gold = sub["text"], sub["intent"]

raw = predict(texts, BASE_SYSTEM)
pred = [parse_intent(r) for r in raw]

valid_json  = sum(p is not None for p in pred) / len(pred)
valid_label = sum(p in set(label_names) for p in pred) / len(pred)
acc         = sum(p == g for p, g in zip(pred, gold)) / len(pred)

print(f"Valid JSON:   {valid_json:.1%}")
print(f"Valid label:  {valid_label:.1%}")
print(f"Accuracy:     {acc:.1%}")

print("\nSample outputs:")
for t, r, g in list(zip(texts, raw, gold))[:5]:
    print(f"- {t}\n  model: {r.strip()}\n  gold:  {g}\n")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Valid JSON:   100.0%
Valid label:  76.5%
Accuracy:     17.5%

Sample outputs:
- I was at the store earlier and my virtual card didn't work. What's the solution for this problem?
  model: {"intent": "solution"}
  gold:  virtual_card_not_working

- How long does a transfer stay pending?
  model: {"intent": "pending_transfer"}
  gold:  pending_transfer

- Help!  My wallet was stolen and someone is taking money out.  I need this money!  What can I do?
  model: {"intent": "contactless_not_working"}
  gold:  cash_withdrawal_not_recognised

- Are there any documents needed for the identity check?
  model: {"intent": "verify_my_identity"}
  gold:  verify_my_identity

- Hello- It is urgent I get a refund on this product. The merchant has not been helpful, what can I do now to get this resolved ASAP?
  model: {"intent": "Refund_not_showing_up"}
  gold:  request_refund



In [11]:
import gc, torch
from transformers import AutoTokenizer

# Free the baseline model from GPU memory
try:
    del model
except NameError:
    pass
gc.collect(); torch.cuda.empty_cache()

tok_train = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct", padding_side="right")

def to_prompt_completion(ex):
    return {
        "prompt": ex["messages"][:2],       # system + user
        "completion": ex["messages"][2:],   # assistant JSON answer
    }

train_pc = data["train"].map(to_prompt_completion, remove_columns=data["train"].column_names)
val_pc   = data["validation"].map(to_prompt_completion, remove_columns=data["validation"].column_names)
print(train_pc[0])

# The token-length check from earlier
lens = [len(tok_train(tok_train.apply_chat_template(m["messages"], tokenize=False)).input_ids)
        for m in data["train"].select(range(500))]
print("\nTokens per example -> mean:", round(sum(lens)/len(lens), 1), "| max:", max(lens))

Map:   0%|          | 0/8993 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

{'prompt': [{'role': 'system', 'content': 'You are a banking assistant. Classify the customer\'s message into one intent. Reply only with JSON in the form {"intent": "<intent_name>"}.'}, {'role': 'user', 'content': 'how do i setup my apple pay'}], 'completion': [{'role': 'assistant', 'content': '{"intent": "apple_pay_or_google_pay"}'}]}

Tokens per example -> mean: 68.6 | max: 136


In [12]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from transformers import AutoModelForCausalLM

base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct", dtype=torch.float32, device_map="cuda"
)

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules="all-linear", task_type="CAUSAL_LM",
)

cfg = SFTConfig(
    output_dir="qwen-bank-intent-lora",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,
    max_length=256,
    logging_steps=25,
    eval_strategy="epoch",
    save_strategy="epoch",
    report_to="none",
    seed=42,
)

trainer = SFTTrainer(
    model=base_model, args=cfg,
    train_dataset=train_pc, eval_dataset=val_pc,
    peft_config=lora, processing_class=tok_train,
)
trainer.model.print_trainable_parameters()
trainer.train()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

TypeError: SFTConfig.__init__() got an unexpected keyword argument 'warmup_ratio'

In [18]:
import gc, torch
try:
    del base_model
except NameError:
    pass
gc.collect(); torch.cuda.empty_cache()

from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from transformers import AutoModelForCausalLM

base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct", dtype=torch.float32, device_map="cuda"
)

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules="all-linear", task_type="CAUSAL_LM",
)

cfg = SFTConfig(
    output_dir="qwen-bank-intent-lora",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=0.05,
    fp16=True,
    max_length=256,
    logging_steps=25,
    eval_strategy="epoch",
    save_strategy="epoch",
    report_to="none",
    seed=42,
)

trainer = SFTTrainer(
    model=base_model, args=cfg,
    train_dataset=train_pc, eval_dataset=val_pc,
    peft_config=lora, processing_class=tok_train,
)
trainer.model.print_trainable_parameters()
trainer.train()


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Tokenizing train dataset:   0%|          | 0/8993 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/8993 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/8993 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/8993 [00:00<?, ? examples/s]

...

Dropping fully masked examples from eval dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,0.036359,0.043746,0.039169,0.988256,619851.000000
2,0.014222,0.028272,0.024442,0.992207,1239702.000000


TrainOutput(global_step=1126, training_loss=0.08825905207454307, metrics={'train_runtime': 699.0895, 'train_samples_per_second': 25.728, 'train_steps_per_second': 1.611, 'total_flos': 3678147437760000.0, 'train_loss': 0.08825905207454307, 'epoch': 2.0})

In [15]:
!pip uninstall -y -q torchao
import importlib
importlib.invalidate_caches()

In [17]:
   !pip install -q -U torchao

In [19]:
import torch
from collections import Counter
from sklearn.metrics import f1_score

# 1. Save the adapter (small: tens of MB)
trainer.save_model("qwen-bank-intent-lora")
tok_train.save_pretrained("qwen-bank-intent-lora")

# 2. Prepare the fine-tuned model for generation
ft_model = trainer.model
ft_model.eval()

@torch.no_grad()
def predict_ft(texts, bs=64):
    outs = []
    for i in range(0, len(texts), bs):
        prompts = [make_prompt(t, SYSTEM) for t in texts[i:i+bs]]   # short prompt, no label list
        enc = tok(prompts, return_tensors="pt", padding=True).to("cuda")
        with torch.autocast("cuda", dtype=torch.float16):
            gen = ft_model.generate(**enc, max_new_tokens=30, do_sample=False)
        outs += tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return outs

def score(texts, gold):
    raw = predict_ft(texts)
    pred = [parse_intent(r) for r in raw]
    valid_json  = sum(p is not None for p in pred) / len(pred)
    valid_label = sum(p in set(label_names) for p in pred) / len(pred)
    acc = sum(p == g for p, g in zip(pred, gold)) / len(pred)
    f1 = f1_score(gold, [p if p else "INVALID" for p in pred], average="macro")
    return raw, pred, valid_json, valid_label, acc, f1

# 3. Same 200 samples as the baseline
_, _, vj, vl, acc, f1 = score(texts, gold)
print("=== Same 200 samples as baseline ===")
print(f"Valid JSON: {vj:.1%} | Valid label: {vl:.1%} | Accuracy: {acc:.1%} | Macro-F1: {f1:.3f}")

# 4. Full test set
full_texts, full_gold = data["test"]["text"], data["test"]["intent"]
raw_f, pred_f, vj, vl, acc, f1 = score(full_texts, full_gold)
print("\n=== Full test set (3,076) ===")
print(f"Valid JSON: {vj:.1%} | Valid label: {vl:.1%} | Accuracy: {acc:.1%} | Macro-F1: {f1:.3f}")

# 5. Most common mistakes
errs = Counter((g, p) for g, p in zip(full_gold, pred_f) if g != p)
print("\nTop 10 confusions (gold -> predicted):")
for (g, p), n in errs.most_common(10):
    print(f"{n:3d}  {g}  ->  {p}")

=== Same 200 samples as baseline ===
Valid JSON: 100.0% | Valid label: 100.0% | Accuracy: 91.5% | Macro-F1: 0.892

=== Full test set (3,076) ===
Valid JSON: 100.0% | Valid label: 99.9% | Accuracy: 92.7% | Macro-F1: 0.893

Top 10 confusions (gold -> predicted):
  6  pending_transfer  ->  transfer_not_received_by_recipient
  5  card_arrival  ->  card_delivery_estimate
  5  top_up_reverted  ->  top_up_failed
  4  card_payment_not_recognised  ->  compromised_card
  4  balance_not_updated_after_bank_transfer  ->  transfer_not_received_by_recipient
  3  extra_charge_on_statement  ->  card_payment_fee_charged
  3  card_delivery_estimate  ->  card_arrival
  3  top_up_by_bank_transfer_charge  ->  transfer_fee_charged
  3  card_payment_not_recognised  ->  direct_debit_payment_not_recognised
  3  compromised_card  ->  lost_or_stolen_card


In [20]:
from huggingface_hub import HfApi

USER = "Kishorekumar32"
MODEL_REPO = f"{USER}/qwen2.5-0.5b-banking77-intent-lora"
DATA_REPO  = f"{USER}/banking77-intent-chat"

# 1. Push the dataset (train / validation / test in chat format)
data.push_to_hub(DATA_REPO)

# 2. Write the model card
card = """---
base_model: Qwen/Qwen2.5-0.5B-Instruct
library_name: peft
license: apache-2.0
datasets:
- mteb/banking77
- __DATA_REPO__
language:
- en
pipeline_tag: text-generation
tags:
- lora
- peft
- trl
- sft
- text-classification
- banking
- intent-classification
---

# Qwen2.5-0.5B Banking77 Intent Classifier (LoRA)

A LoRA adapter for `Qwen/Qwen2.5-0.5B-Instruct`, fine-tuned to classify customer banking queries into one of 77 intents and reply in strict JSON, for example `{"intent": "card_arrival"}`.

## Results (official Banking77 test set, 3,076 examples)

| Metric | Base model* | Fine-tuned |
|---|---|---|
| Accuracy | 17.5% | **92.7%** |
| Macro-F1 | not measured | 0.893 |
| Valid JSON | 100% | 100% |
| Valid intent label | 76.5% | 99.9% |

*Base model was evaluated on a fixed random subset of 200 test examples, with all 77 valid intent names listed in the prompt. The fine-tuned model scored 91.5% on the same 200 examples and uses a shorter prompt with no label list.

## Training

- Method: LoRA (r=16, alpha=32, dropout=0.05) on all linear layers; 8.8M trainable parameters (1.75% of 502.8M)
- Data: 8,993 train / 1,000 validation (stratified split from Banking77 train), tested on the official 3,076-example test set
- Loss computed on the assistant answer only
- 2 epochs, batch size 16, learning rate 2e-4 (cosine schedule), fp16, seed 42
- Hardware: single Google Colab T4 GPU, about 11.5 minutes
- Libraries: transformers, peft, trl

## Usage

```python
import torch, json
from transformers import AutoTokenizer
from peft import AutoPeftModelForCausalLM

repo = "__MODEL_REPO__"
tok = AutoTokenizer.from_pretrained(repo, padding_side="left")
model = AutoPeftModelForCausalLM.from_pretrained(repo, dtype=torch.float16, device_map="auto")

SYSTEM = ("You are a banking assistant. Classify the customer's message into one "
          "intent. Reply only with JSON in the form {\\"intent\\": \\"<intent_name>\\"}.")
msgs = [{"role": "system", "content": SYSTEM},
        {"role": "user", "content": "My card still has not arrived"}]
prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
inputs = tok(prompt, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=30, do_sample=False)
print(tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
```

## Limitations

- Most errors are between semantically close intents (for example `card_arrival` vs `card_delivery_estimate`, `top_up_reverted` vs `top_up_failed`, `compromised_card` vs `lost_or_stolen_card`).
- Trained and tested only on short English banking queries. It will always pick one of the 77 known intents, even for out-of-scope messages.
- Not intended for production use in real banking systems without further validation.

## Data and licenses

Banking77 (Casanueva et al., 2020), CC BY 4.0. Base model: Apache 2.0.
"""
card = card.replace("__MODEL_REPO__", MODEL_REPO).replace("__DATA_REPO__", DATA_REPO)
with open("qwen-bank-intent-lora/README.md", "w") as f:
    f.write(card)

# 3. Upload the adapter, skipping checkpoint folders
api = HfApi()
api.create_repo(MODEL_REPO, exist_ok=True)
api.upload_folder(
    folder_path="qwen-bank-intent-lora",
    repo_id=MODEL_REPO,
    ignore_patterns=["checkpoint-*", "runs/*"],
)

print("Model:  https://huggingface.co/" + MODEL_REPO)
print("Data:   https://huggingface.co/datasets/" + DATA_REPO)

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /tmp/tmpk79y3sey.parquet    : 100%|##########|  185kB /  185kB            

Validating                    :           |   0%            

Model:  https://huggingface.co/Kishorekumar32/qwen2.5-0.5b-banking77-intent-lora
Data:   https://huggingface.co/datasets/Kishorekumar32/banking77-intent-chat


In [21]:
import gc, torch

# 1. Delete every local copy of the model and free the GPU
for name in ["trainer", "ft_model", "base_model", "model"]:
    if name in globals():
        del globals()[name]
gc.collect(); torch.cuda.empty_cache()
print("GPU memory in use (GB):", round(torch.cuda.memory_allocated() / 1e9, 2))

# 2. Load fresh from the Hub, exactly as a stranger would
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer

repo = "Kishorekumar32/qwen2.5-0.5b-banking77-intent-lora"
tok_hub = AutoTokenizer.from_pretrained(repo, padding_side="left")
hub_model = AutoPeftModelForCausalLM.from_pretrained(repo, dtype=torch.float16, device_map="cuda")
hub_model.eval()

# 3. Re-run the same 200 samples
@torch.no_grad()
def predict_hub(texts, bs=64):
    outs = []
    for i in range(0, len(texts), bs):
        prompts = [make_prompt(t, SYSTEM) for t in texts[i:i+bs]]
        enc = tok_hub(prompts, return_tensors="pt", padding=True).to("cuda")
        gen = hub_model.generate(**enc, max_new_tokens=30, do_sample=False)
        outs += tok_hub.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return outs

pred = [parse_intent(r) for r in predict_hub(texts)]
acc = sum(p == g for p, g in zip(pred, gold)) / len(gold)
print(f"Accuracy from Hub model (same 200 samples): {acc:.1%}")

# 4. One live demo
demo = predict_hub(["My card still has not arrived, it has been two weeks"])
print("Demo output:", demo[0])

GPU memory in use (GB): 2.0


tokenizer_config.json:   0%|          | 0.00/721 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/2.51k [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/1.18k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 35.2MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

Accuracy from Hub model (same 200 samples): 91.5%
Demo output: {"intent": "card_arrival"}
